In [77]:
import os
import numpy as np
import tensorflow as tf
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from hand_processing import process_images
from tensorflow.keras.callbacks import EarlyStopping

BASE_DIR = "ASL_Alphabet_Dataset/asl_alphabet_train/"
MAX_IMAGES = 2000
TEST_SIZE = 0.2
BATCH_SIZE = 32
EPOCHS = 200


In [78]:
X, y = process_images(BASE_DIR, MAX_IMAGES)
np.save('features.npy', X)
np.save('labels.npy', y)

I0000 00:00:1736903497.544837   69921 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1736903497.545753  238924 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 22.3.6), renderer: Mesa Intel(R) Xe Graphics (TGL GT2)
W0000 00:00:1736903497.558136  238917 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1736903497.573497  238921 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


KeyboardInterrupt: 

In [79]:
X = np.load('features.npy')
y = np.load('labels.npy')
print(X.shape)
print(y.shape)

(36880, 69)
(36880,)


In [80]:
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(y)
joblib.dump(label_encoder, 'label_encoder.pkl')

['label_encoder.pkl']

In [83]:
X_train, X_test, y_train, y_test = train_test_split(X, encoded_labels, test_size=TEST_SIZE, random_state=42)


early_stopping = EarlyStopping(
    monitor='val_loss',  
    patience=20,          
    restore_best_weights=True  
)

model = tf.keras.Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(np.unique(encoded_labels)), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping] 
)

Epoch 1/200
922/922 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6391 - loss: 1.3696 - val_accuracy: 0.9264 - val_loss: 0.3071
Epoch 2/200
922/922 ━━━━━━━━━━━━━━━━━━━━ 1s 991us/step - accuracy: 0.9044 - loss: 0.3619 - val_accuracy: 0.9492 - val_loss: 0.2082
Epoch 3/200
922/922 ━━━━━━━━━━━━━━━━━━━━ 1s 990us/step - accuracy: 0.9320 - loss: 0.2578 - val_accuracy: 0.9555 - val_loss: 0.1692
Epoch 4/200
922/922 ━━━━━━━━━━━━━━━━━━━━ 1s 985us/step - accuracy: 0.9404 - loss: 0.2236 - val_accuracy: 0.9595 - val_loss: 0.1526
Epoch 5/200
922/922 ━━━━━━━━━━━━━━━━━━━━ 1s 975us/step - accuracy: 0.9480 - loss: 0.1972 - val_accuracy: 0.9648 - val_loss: 0.1323
Epoch 6/200
922/922 ━━━━━━━━━━━━━━━━━━━━ 1s 987us/step - accuracy: 0.9554 - loss: 0.1735 - val_accuracy: 0.9669 - val_loss: 0.1219
Epoch 7/200
922/922 ━━━━━━━━━━━━━━━━━━━━ 1s 964us/step - accuracy: 0.9564 - loss: 0.1611 - val_accuracy: 0.9714 - val_loss: 0.1106
Epoch 8/200
922/922 ━━━━━━━━━━━━━━━━━━━━ 1s 968us/step - accuracy: 0.9614 - loss: 0.1

In [84]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=2)
print(f"Test Loss: {loss}, Test Accuracy: {accuracy}")

y_pred = np.argmax(model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

231/231 - 0s - 806us/step - accuracy: 0.9877 - loss: 0.0514
Test Loss: 0.05141361430287361, Test Accuracy: 0.987662672996521
231/231 ━━━━━━━━━━━━━━━━━━━━ 0s 582us/step
              precision    recall  f1-score   support

           A       0.97      0.99      0.98       235
           B       1.00      1.00      1.00       297
           C       0.98      0.99      0.99       236
           D       1.00      0.99      0.99       318
           E       0.99      0.99      0.99       283
           F       1.00      0.99      1.00       387
           G       0.99      1.00      0.99       320
           H       1.00      0.99      1.00       319
           I       0.98      0.98      0.98       290
           J       0.99      0.98      0.99       305
           K       0.99      0.99      0.99       340
           L       1.00      1.00      1.00       342
           M       0.93      0.95      0.94       195
           N       0.96      0.92      0.94       143
           O       0.

In [40]:
model.save("ASL_model.keras")